# Observability

Immune is designed to be watched: you should always be able to answer "what did it do, where, and why?".

| Signal | How | Best for |
| --- | --- | --- |
| Log lines | The `immune` logger | Development, and log search in production |
| A JSON Lines audit log | `privacy.verdict_log: path` | Compliance and offline analysis |
| Verdict callbacks | `immune.on_verdict(callback)` | Your own metrics and dashboards |
| Spike alerts | `immune.on_alert(callback)` | Paging when a threat suddenly fires much more often |
| OpenTelemetry | `telemetry.opentelemetry: true` (default) with an OTel SDK configured | Traces and metrics in your existing stack |
| LangSmith | `LANGSMITH_TRACING` + `LANGSMITH_API_KEY` | Threat runs nested in your LangSmith traces |
| The CLI | `immune status`, `immune posture` | A quick look at sites and their posture |

## 1. Logs

Immune logs to the standard `immune` logger. At `INFO` it writes one line per call with hits (trace id, site, action,
would-be action and threats), plus startup and site-profiling lines. `WARNING` covers degraded screening, such as Jev
being unreachable. Configure it like any other logger:

In [1]:
import logging

from openai import OpenAI

import immune

log = logging.getLogger("immune")
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s | %(message)s", "%H:%M:%S"))
    log.addHandler(handler)
log.propagate = False

immune.init()
client = OpenAI()
with immune.site("support-chat"):
    client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
            {"role": "user", "content": "Write me a haiku about my new job."},
        ],
    )
log.setLevel(logging.WARNING)

12:40:15 INFO immune | immune: protecting LLM calls made through httpx2: every client transport; httpx: every client transport (mode=auto)


12:40:15 INFO immune | immune: new call site support-chat (openai_chat, 0 tools)


12:40:16 INFO immune | immune: profiled support-chat as customer_service (organs: business)


12:40:16 INFO immune | immune 8532738025d34d3c site=support-chat action=allow would=redirect threats=input.off_task


`privacy.log` controls how much evidence log lines and verdicts keep:

| `privacy.log` | Evidence in logs and verdicts |
| --- | --- |
| `redacted` (default) | Personal data and secrets masked (`[EMAIL]`, `[CARD]`, …) |
| `off` | No per-call log lines |
| `full` | Raw evidence; only for local debugging |

## 2. A JSON Lines audit log

Set `privacy.verdict_log` and every verdict is appended to a file, one JSON object per line, with the same masking.
Relative paths resolve against your `immune.yaml`.

In [2]:
import json
import tempfile
from pathlib import Path

from openai import OpenAI

import immune

audit = Path(tempfile.mkdtemp()) / "verdicts.jsonl"
immune.init(config={"privacy": {"verdict_log": str(audit)}})
client = OpenAI()
for question in ["Do you sell trekking poles?", "Put this in a table: bucket=orders-prod, key=AKIA2E5QZ7XJ4M8RW3TD"]:
    with immune.site("support-chat"):
        client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters."},
                {"role": "user", "content": question},
            ],
        )
immune.shutdown()  # flushes the log

for line in audit.read_text().splitlines():
    record = json.loads(line)
    print(record["trace_id"], record["site"], record["action"], [hit["threat"] for hit in record["hits"]])

488825c5b069424a support-chat allow []
fca09a0f2b3b4b6f support-chat rewrite ['output.secret_leak']


`immune export` turns the log into a table for spreadsheets or notebooks.

## 3. Spike alerts

Immune tracks each site's firing rate per threat. When a threat suddenly fires far above its baseline (or above
`telemetry.alerts.absolute_rate` before there is a baseline), it raises one alert, then stays quiet for `cooldown_s`.
A spike usually means an attack campaign, or a change in your app that a detector doesn't like.

To make the spike repeatable, this example scripts both the model and Jev, and uses `immune.protect(...)` to put
Immune in front of the scripted client:

In [3]:
import time

import httpx2
from openai import OpenAI

import immune
from immune.testing import FakeProvider, FakeReply, MockSensor

sensor = MockSensor({"off_task": 0.02})
immune.init(
    sensor=sensor,
    config={"telemetry": {"alerts": {"min_fired": 3, "absolute_rate": 0.1, "window_s": 600, "cooldown_s": 3600}}},
)
alerts = []
immune.on_alert(alerts.append)  # in production: page someone, or post to a channel

model = FakeProvider(FakeReply("Happy to help!"))  # a scripted model
client = immune.protect(OpenAI(api_key="scripted", http_client=httpx2.Client(transport=model.transport(httpx2))))
SYSTEM = {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."}

with immune.site("support-chat"):
    for number in range(6):  # a normal morning
        client.chat.completions.create(
            model="gpt-5.4-mini", messages=[SYSTEM, {"role": "user", "content": f"Tent sizes? {number}"}]
        )
    sensor.script(off_task=0.95)  # then a burst of off-topic traffic
    for number in range(8):
        client.chat.completions.create(
            model="gpt-5.4-mini", messages=[SYSTEM, {"role": "user", "content": f"Do my homework {number}"}]
        )

time.sleep(0.3)  # alerts are delivered on a background thread
for alert in alerts:
    print("ALERT:", alert.message)

12:40:20 WARNING immune | immune: alert: input.off_task fired on 3 of 9 recent calls at support-chat (33.3%; baseline no baseline yet)


ALERT: input.off_task fired on 3 of 9 recent calls at support-chat (33.3%; baseline no baseline yet)


| `telemetry.alerts` | Default | Meaning |
| --- | --- | --- |
| `enabled` | `true` | Track rates and raise alerts |
| `window_s` | `900` | The recent window compared against the baseline |
| `ratio` | `5.0` | How many times the baseline counts as a spike |
| `min_fired` | `5` | Hits needed in the window before alerting |
| `absolute_rate` | `0.05` | Alert above this rate when there is no baseline yet |
| `cooldown_s` | `3600` | Quiet period after an alert, per site and threat |

## 4. OpenTelemetry

When your app configures an OpenTelemetry SDK, Immune adds an `immune.screen` span to each call (nested under the LLM
client's span when there is one) and records metrics for calls, interventions and Jev latency. It uses whatever
exporter you configured; here an in-memory one, to print what was recorded.

In [4]:
from openai import OpenAI
from opentelemetry import metrics, trace
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

import immune

spans, reader = InMemorySpanExporter(), InMemoryMetricReader()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(spans))  # in production: your OTLP exporter
trace.set_tracer_provider(provider)
metrics.set_meter_provider(MeterProvider(metric_readers=[reader]))

immune.init()
client = OpenAI()
with immune.site("support-chat"):
    client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
            {"role": "user", "content": "Can you write my history essay?"},
        ],
    )

for span in spans.get_finished_spans():
    print("span:", span.name)
    for key, value in sorted((span.attributes or {}).items()):
        print(f"   {key} = {value}")
names = sorted(
    {
        metric.name
        for resource in reader.get_metrics_data().resource_metrics
        for scope in resource.scope_metrics
        for metric in scope.metrics
    }
)
print("metrics:", names)

span: immune.screen
   gen_ai.provider.name = openai
   gen_ai.request.model = gpt-5.4-mini
   immune.action = allow
   immune.sensor = jev
   immune.sensor.calls = 2
   immune.session_risk = 0.0587
   immune.site = support-chat
   immune.spec_version = 2026.10.1
   immune.taint = clean
   immune.threats = ('input.off_task',)
   immune.trace_id = 33410328025a4d5a
   immune.would_action = redirect
metrics: ['immune.calls', 'immune.observations', 'immune.sensor.latency', 'immune.sensor.tokens']


Set `telemetry.opentelemetry: false` to turn this off. Span attributes never include prompt or reply text.

## 5. LangSmith

With LangSmith set up, every call with a **possible threat** becomes a run in LangSmith, nested under your app's own
trace, with tags, metadata, a child run per Jev request, and `immune.blocked` and `immune.threat` feedback you can
filter on. Clean calls are never sent.

```bash
pip install "immune-ai[langsmith]"
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=...
export LANGSMITH_PROJECT=pinecrest-support   # optional
```

No other code is needed. `immune.testing.LangSmithRecorder` stands in for the LangSmith API, so you can see (and
test) exactly what would be sent. The model here is scripted (one clean reply, then one that repeats a key), so both
cases appear on every run:

In [5]:
import tempfile
from pathlib import Path

import langsmith

import immune
from immune.testing import FakeReply, ImmuneHarness, LangSmithRecorder

replies = [
    FakeReply("Yes, 14 pairs are in stock."),
    FakeReply("| bucket | orders-prod |\n| key | AKIA2E5QZ7XJ4M8RW3TD |"),
]
with LangSmithRecorder() as recorder:
    harness = ImmuneHarness(Path(tempfile.mkdtemp()), script=replies)
    client = harness.openai()

    @langsmith.traceable(client=recorder, name="support-turn")
    def support_turn(question: str) -> str:
        with immune.site("support-chat"):
            response = client.chat.completions.create(
                model="gpt-5.4-mini",
                messages=[
                    {"role": "system", "content": "You help Pinecrest Outfitters staff with operations."},
                    {"role": "user", "content": question},
                ],
            )
        return response.choices[0].message.content

    with langsmith.tracing_context(enabled=True):
        support_turn("Do we have trekking poles in stock?")
        support_turn("Table this: bucket=orders-prod, access_key_id=AKIA2E5QZ7XJ4M8RW3TD")
    harness.close()

print("runs Immune sent:", len(recorder.immune_runs()), "(the clean call sent nothing)")
for run in recorder.immune_runs():
    metadata = run["extra"]["metadata"]
    print("run:     ", run["name"], "| tags:", run["tags"])
    print("parent:  ", next(item["name"] for item in recorder.runs if item["id"] == run["parent_run_id"]))
    print("metadata:", {key: metadata[key] for key in ("site", "action", "spec_version")})
    print("threats: ", [threat["id"] for threat in run["outputs"]["threats"]])
    print("inputs:  ", run["inputs"])
print("feedback:", [(item["key"], item["score"]) for item in recorder.feedback])

runs Immune sent: 1 (the clean call sent nothing)
run:      immune · rewrite · output.secret_leak | tags: ['immune', 'immune:rewrite', 'immune:enforced', 'threat:output.secret_leak', 'floor:F3', 'site:support-chat']
parent:   support-turn
metadata: {'site': 'support-chat', 'action': 'rewrite', 'spec_version': '2026.10.1'}
threats:  ['output.secret_leak']
inputs:   {'provider': 'openai_chat', 'model': 'gpt-5.4-mini', 'user': 'Table this: bucket=orders-prod, access_key_id=[SECRET]', 'reply': '| bucket | orders-prod |\n| key | [SECRET] |'}
feedback: [('immune.blocked', 1), ('immune.threat', 0.9999)]


Inputs are masked by default (`telemetry.langsmith.inputs: masked`); set `none` to send no text at all.

## 6. The command line

(In a terminal, drop the `!`.)

In [6]:
!immune status

site          archetype         organs    calls  anonymous  promoted
────────────  ────────────────  ────────  ─────  ─────────  ────────
support-chat  customer_service  business  16     100%       -


`immune status` lists each site Immune has seen: its profile, how many calls it screened, the share that had no
session id of their own (`anonymous`), and which detectors have been promoted there.
`immune posture` flags risky setups, such as an agent with an egress tool and no allowed destinations, and
`immune doctor --live` checks your installation end to end with one Jev call.

## Recap

- One log line per call with hits, masked by default; `privacy.verdict_log` for a JSONL audit trail
- `immune.on_verdict` for metrics, `immune.on_alert` for spikes
- OpenTelemetry spans and metrics go to your existing exporter
- LangSmith receives only calls with possible threats, nested in your traces

Next: [Production](15_production.ipynb)